# Bài tập Buổi 10: Singular Value Decomposition (SVD)

**Họ và tên:** Mai Gia Bảo
**MSSV:** 24520163

Bài tập gồm hai câu. Câu 1 phân rã một ma trận rank thấp bằng SVD với đầy đủ các bước. Câu 2 chọn số thành phần k cho Truncated SVD dựa trên tỷ lệ năng lượng. Mỗi yêu cầu được trình bày bằng một ô văn bản nêu các bước và một ô code kiểm tra bằng Python.

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
print("Đã sẵn sàng.")

# Câu 1. Phân rã ma trận A

Ma trận đầu vào:

$$A=\begin{bmatrix}1&0&1\\0&1&1\\1&0&1\\0&1&1\end{bmatrix}\in\mathbb{R}^{4\times3}$$

In [ ]:
A = np.array([[1, 0, 1],
              [0, 1, 1],
              [1, 0, 1],
              [0, 1, 1]], dtype=float)
print("A =")
print(A)
print("Kích thước A:", A.shape)

## Yêu cầu 1. Cột phụ thuộc tuyến tính và rank(A)

Gọi ba cột của A là $c_1, c_2, c_3$:

$$c_1=(1,0,1,0)^\top,\quad c_2=(0,1,0,1)^\top,\quad c_3=(1,1,1,1)^\top$$

Nhận thấy $c_3=c_1+c_2$ nên ba cột phụ thuộc tuyến tính. Hai cột $c_1, c_2$ độc lập tuyến tính vì không cột nào là bội của cột kia. Vậy số cột độc lập là 2, suy ra $\operatorname{rank}(A)=2$.

In [ ]:
c1, c2, c3 = A[:, 0], A[:, 1], A[:, 2]
print("c1 + c2 =", c1 + c2)
print("c3      =", c3)
print("c3 bằng c1 cộng c2:", np.allclose(c3, c1 + c2))
print("rank(A) =", np.linalg.matrix_rank(A))

## Yêu cầu 2. Tính $A^\top A$

Phần tử $(i,j)$ của $A^\top A$ bằng tích vô hướng $c_i\cdot c_j$:

$$A^\top A=\begin{bmatrix}2&0&2\\0&2&2\\2&2&4\end{bmatrix}$$

In [ ]:
AtA = A.T @ A
print("A^T A =")
print(AtA)

## Yêu cầu 3. Eigenvalues và eigenvectors trực chuẩn của $A^\top A$

Giải phương trình đặc trưng $\det(A^\top A-\lambda I)=0$. Khai triển định thức thu được:

$$\det(A^\top A-\lambda I)=(2-\lambda)\,\lambda\,(\lambda-6)$$

Vậy các eigenvalue là $\lambda=6,\ \lambda=2,\ \lambda=0$. Giải $(A^\top A-\lambda I)v=0$ cho từng eigenvalue rồi chuẩn hóa độ dài về 1:

$$v_{(\lambda=6)}=\tfrac{1}{\sqrt6}(1,1,2)^\top,\quad v_{(\lambda=2)}=\tfrac{1}{\sqrt2}(1,-1,0)^\top,\quad v_{(\lambda=0)}=\tfrac{1}{\sqrt3}(1,1,-1)^\top$$

Ba vector này trực giao từng đôi và có độ dài 1 nên tạo thành hệ trực chuẩn.

In [ ]:
eigvals, eigvecs = np.linalg.eigh(AtA)   # eigh cho eigenvalue theo thứ tự tăng dần
print("Eigenvalues (tăng dần):", eigvals)
print("Eigenvectors (theo cột):")
print(eigvecs)
print("V^T V gần I nên các eigenvector trực chuẩn:", np.allclose(eigvecs.T @ eigvecs, np.eye(3)))

## Yêu cầu 4. Sắp xếp eigenvalue giảm dần

$$\lambda_1=6\ \ge\ \lambda_2=2\ \ge\ \lambda_3=0$$

In [ ]:
order = np.argsort(eigvals)[::-1]
lam = eigvals[order]
print("lambda_1, lambda_2, lambda_3 =", lam)

## Yêu cầu 5. Các singular value

$$\sigma_1=\sqrt{\lambda_1}=\sqrt6\approx2.449,\quad \sigma_2=\sqrt{\lambda_2}=\sqrt2\approx1.414,\quad \sigma_3=\sqrt{\lambda_3}=0$$

In [ ]:
sigmas = np.sqrt(np.clip(lam, 0, None))
print("sigma =", sigmas)
print("Đối chiếu singular value từ numpy:", np.linalg.svd(A, compute_uv=False))

## Yêu cầu 6. Right singular vectors

Right singular vector chính là eigenvector của $A^\top A$ đã sắp theo eigenvalue giảm dần:

$$v_1=\tfrac{1}{\sqrt6}(1,1,2)^\top,\quad v_2=\tfrac{1}{\sqrt2}(1,-1,0)^\top,\quad v_3=\tfrac{1}{\sqrt3}(1,1,-1)^\top$$

In [ ]:
V = eigvecs[:, order].copy()   # sắp cột theo eigenvalue giảm dần
for j in range(3):                # cố định dấu cho dễ đọc
    if V[np.argmax(np.abs(V[:, j])), j] < 0:
        V[:, j] = -V[:, j]
print("v1 =", V[:, 0])
print("v2 =", V[:, 1])
print("v3 =", V[:, 2])

## Yêu cầu 7. Left singular vectors ứng với $\sigma_i>0$

Với $\sigma_1,\sigma_2>0$ dùng công thức $u_i=Av_i/\sigma_i$. Cụ thể $Av_1=\tfrac{1}{\sqrt6}(3,3,3,3)^\top$ nên chia cho $\sigma_1=\sqrt6$ được:

$$u_1=\tfrac{1}{2}(1,1,1,1)^\top,\qquad u_2=\tfrac{1}{2}(1,-1,1,-1)^\top$$

In [ ]:
u1 = A @ V[:, 0] / sigmas[0]
u2 = A @ V[:, 1] / sigmas[1]
print("u1 =", u1)
print("u2 =", u2)
print("Độ dài u1, u2:", np.linalg.norm(u1), np.linalg.norm(u2))

## Yêu cầu 8. Bổ sung vector trực chuẩn và viết Full SVD

Cần thêm $u_3, u_4$ trực chuẩn và vuông góc với $u_1, u_2$ để U đủ 4 cột. Giải điều kiện vuông góc cho:

$$u_3=\tfrac{1}{\sqrt2}(1,0,-1,0)^\top,\qquad u_4=\tfrac{1}{\sqrt2}(0,1,0,-1)^\top$$

Full SVD $A=U\Sigma V^\top$ với $U\in\mathbb{R}^{4\times4}$, $\Sigma\in\mathbb{R}^{4\times3}$ có đường chéo $\sqrt6,\sqrt2,0$, và $V^\top\in\mathbb{R}^{3\times3}$.

In [ ]:
u3 = np.array([1, 0, -1, 0], dtype=float) / np.sqrt(2)
u4 = np.array([0, 1, 0, -1], dtype=float) / np.sqrt(2)
U = np.column_stack([u1, u2, u3, u4])
Sigma = np.zeros((4, 3))
for i in range(3):
    Sigma[i, i] = sigmas[i]
print("U =")
print(U)
print("Sigma =")
print(Sigma)
print("V^T =")
print(V.T)

## Yêu cầu 9. Reduced SVD

Chỉ giữ hai thành phần ứng với singular value dương:

$$A=U_r\Sigma_r V_r^\top,\quad U_r=[u_1\ u_2],\ \Sigma_r=\operatorname{diag}(\sqrt6,\sqrt2),\ V_r^\top=[v_1\ v_2]^\top$$

In [ ]:
r = 2
Ur = U[:, :r]
Sigma_r = np.diag(sigmas[:r])
Vr = V[:, :r]
print("U_r =")
print(Ur)
print("Sigma_r =")
print(Sigma_r)
print("V_r^T =")
print(Vr.T)

## Yêu cầu 10. Kích thước của Reduced SVD

$$U_r\in\mathbb{R}^{4\times2},\qquad \Sigma_r\in\mathbb{R}^{2\times2},\qquad V_r^\top\in\mathbb{R}^{2\times3}$$

In [ ]:
print("U_r:", Ur.shape)
print("Sigma_r:", Sigma_r.shape)
print("V_r^T:", Vr.T.shape)

## Yêu cầu 11. Kiểm tra tính đúng đắn

Kiểm tra $U^\top U=I$, $V^\top V=I$, $U\Sigma V^\top=A$, và Reduced SVD cũng tái tạo đúng A.

In [ ]:
print("U^T U = I:", np.allclose(U.T @ U, np.eye(4)))
print("V^T V = I:", np.allclose(V.T @ V, np.eye(3)))
print("U Sigma V^T = A:", np.allclose(U @ Sigma @ V.T, A))
print("Reduced tái tạo A:", np.allclose(Ur @ Sigma_r @ Vr.T, A))

## Yêu cầu 12. Giải thích

**Vì sao có một singular value bằng 0.** Vì $\operatorname{rank}(A)=2$ nhỏ hơn số cột là 3, ba cột phụ thuộc tuyến tính nên $A^\top A$ suy biến, có một eigenvalue bằng 0 và kéo theo một singular value bằng 0.

**Liên hệ với null space.** Right singular vector $v_3=\tfrac{1}{\sqrt3}(1,1,-1)^\top$ ứng với singular value bằng 0 thỏa $Av_3=0$, nghĩa là $v_3$ nằm trong null space của A và sinh ra null space đó.

**Vì sao Reduced SVD vẫn tái tạo chính xác A.** Thành phần bị bỏ có $\sigma_3=0$ nên số hạng $\sigma_3 u_3 v_3^\top$ bằng 0, không đóng góp gì. Do đó giữ hai thành phần đầu là đủ để tái tạo A một cách chính xác.

In [ ]:
print("A v3 =", A @ V[:, 2], ", gần vector 0:", np.allclose(A @ V[:, 2], 0))
print("sigma_3 =", sigmas[2], ", nên số hạng thứ ba đóng góp 0")
print("Sai số Reduced so với A:", np.linalg.norm(Ur @ Sigma_r @ Vr.T - A))

# Câu 2. Chọn k cho Truncated SVD bằng tỷ lệ năng lượng

Ma trận dữ liệu X có các singular value:

$$\sigma_1=12,\quad \sigma_2=5,\quad \sigma_3=3,\quad \sigma_4=1$$

In [ ]:
sig = np.array([12, 5, 3, 1], dtype=float)
print("Các singular value:", sig)

## Yêu cầu 1. Rank của X

Cả bốn singular value đều dương nên X có đủ bốn thành phần độc lập, suy ra $\operatorname{rank}(X)=4$.

In [ ]:
print("Số singular value dương =", int((sig > 0).sum()))
print("rank(X) =", int((sig > 0).sum()))

## Yêu cầu 2. Tổng năng lượng

$$E_{\text{total}}=\sum_{i=1}^{4}\sigma_i^2=12^2+5^2+3^2+1^2=144+25+9+1=179$$

In [ ]:
E = sig ** 2
E_total = E.sum()
print("Năng lượng từng thành phần:", E)
print("E_total =", E_total)

## Yêu cầu 3. Tỷ lệ năng lượng $R(k)$

$$R(k)=\frac{\sum_{i=1}^{k}\sigma_i^2}{\sum_{i=1}^{4}\sigma_i^2}$$

$$R(1)=\tfrac{144}{179}\approx0.8045,\quad R(2)=\tfrac{169}{179}\approx0.9441,\quad R(3)=\tfrac{178}{179}\approx0.9944,\quad R(4)=1.0$$

In [ ]:
cum = np.cumsum(E)
R = cum / E_total
for k in range(1, 5):
    print(f"R({k}) = {cum[k-1]:.0f}/{E_total:.0f} = {R[k-1]:.4f}")

## Yêu cầu 4. Bảng năng lượng giữ lại và tỷ lệ

| $k$ | Năng lượng giữ lại | Tỷ lệ $R(k)$ |
|---:|---:|---:|
| 1 | 144 | 0.8045 |
| 2 | 169 | 0.9441 |
| 3 | 178 | 0.9944 |
| 4 | 179 | 1.0000 |

In [ ]:
import pandas as pd
bang = pd.DataFrame({"k": [1, 2, 3, 4],
                     "Nang_luong_giu_lai": cum.astype(int),
                     "R(k)": R.round(4)})
print(bang.to_string(index=False))

## Yêu cầu 5. k nhỏ nhất theo ngưỡng năng lượng

* Giữ ít nhất 80 phần trăm: $R(1)=0.8045\ge0.80$ nên $k=1$.
* Giữ ít nhất 90 phần trăm: $R(2)=0.9441\ge0.90$ nên $k=2$.
* Giữ ít nhất 95 phần trăm: $R(2)=0.9441<0.95$ còn $R(3)=0.9944\ge0.95$ nên $k=3$.
* Giữ ít nhất 99 phần trăm: $R(3)=0.9944\ge0.99$ nên $k=3$.

In [ ]:
for thr in [0.80, 0.90, 0.95, 0.99]:
    k = int(np.argmax(R >= thr)) + 1
    print(f"Ngưỡng {int(thr*100)} phần trăm, k nhỏ nhất = {k} (R = {R[k-1]:.4f})")

## Yêu cầu 6. Sai số Frobenius của xấp xỉ hạng k

$$\|X-X_k\|_F=\sqrt{\sum_{i=k+1}^{4}\sigma_i^2}$$

$$\|X-X_1\|_F=\sqrt{35}\approx5.916,\quad \|X-X_2\|_F=\sqrt{10}\approx3.162,\quad \|X-X_3\|_F=\sqrt{1}=1.0$$

In [ ]:
for k in [1, 2, 3]:
    err = np.sqrt(E[k:].sum())
    print(f"||X - X_{k}||_F = sqrt({E[k:].sum():.0f}) = {err:.4f}")

## Yêu cầu 7. Bảng tỷ lệ năng lượng và sai số

| $k$ | $R(k)$ | $\|X-X_k\|_F$ |
|---:|---:|---:|
| 1 | 0.8045 | 5.9161 |
| 2 | 0.9441 | 3.1623 |
| 3 | 0.9944 | 1.0000 |

In [ ]:
bang2 = pd.DataFrame({"k": [1, 2, 3],
                      "R(k)": R[:3].round(4),
                      "Frobenius": [round(float(np.sqrt(E[k:].sum())), 4) for k in [1, 2, 3]]})
print(bang2.to_string(index=False))

## Yêu cầu 8. Chọn k khi $\sigma_4$ chủ yếu là nhiễu

Chọn $k=3$. Bỏ thành phần thứ tư loại được nhiễu $\sigma_4=1$ mà vẫn giữ $R(3)\approx0.9944$ tức khoảng 99.44 phần trăm năng lượng, sai số chỉ còn $\|X-X_3\|_F=1.0$. Đây là mức nén tốt vì loại nhiễu nhưng gần như không mất tín hiệu.

In [ ]:
k_choose = 3
print(f"Chọn k = {k_choose}, giữ {R[k_choose-1]*100:.2f} phần trăm năng lượng, sai số {np.sqrt(E[k_choose:].sum()):.4f}")

## Yêu cầu 9. Trả lời câu hỏi khái niệm

**Vì sao singular value phải sắp xếp giảm dần.** Để mỗi lần cắt bớt thì phần giữ lại luôn là các thành phần mang nhiều năng lượng nhất, nhờ vậy tỷ lệ $R(k)$ tăng theo k và xấp xỉ hạng k là tốt nhất có thể với mỗi k.

**Giữ càng nhiều singular value có luôn tốt hơn không.** Không phải, giữ thêm các thành phần nhỏ như $\sigma_4$ chủ yếu thêm nhiễu và làm giảm mức nén, nên đôi khi bỏ bớt lại cho kết quả sạch hơn và gọn hơn.

**Đánh đổi giữa tỷ lệ năng lượng và mức nén.** Giữ nhiều thành phần thì tỷ lệ năng lượng cao và tái tạo sát dữ liệu gốc nhưng tốn bộ nhớ và ít nén, giữ ít thành phần thì nén mạnh và gọn nhưng mất thông tin, cần cân bằng theo yêu cầu bài toán.

In [ ]:
print("R(k) theo k:", R)
print("R không giảm khi k tăng:", bool(np.all(np.diff(R) >= 0)))